# Metajeux-04 : Le chemin minimal, témoin construit et vérifié indépendamment — de Robinson-Goforth au vérificateur séparé

**Navigation** : [Version Python](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) | [Index](README.md)

**La loi que ce notebook met à l'épreuve** — Loi II du deuxième voyage de digestion ICT :

> *recoordonner + passer du vérificateur au constructeur* : le précédent important n'est pas seulement
> le changement de coordonnées ; c'est changement de coordonnées **plus** passage du vérificateur au constructeur.

Elle est franchie une fois sur le substrat Life (GOL-01 : la machine **redécouvre** le glider au lieu
de le recopier). Ce notebook exige la même loi sur un **substrat totalement différent** :
l'univers fini des jeux 2×2 ordinaux de Robinson-Goforth. Si la loi ne tient que sur Life, ce n'est pas une loi.

Ici, le vérificateur c'est le solveur qui **reconnaît** qu'un chemin d'échanges élémentaires relie deux jeux.
Le constructeur, c'est la machine qui **produit** ce chemin — puis un **vérificateur séparé** qui vérifie,
indépendamment, que le chemin est valide **et minimal**. Le chemin produit est un **témoin construit et
vérifié indépendamment**, pas un certificat au sens formel : dans un univers fini et énumérable, la
minimalité se prouve par recherche exhaustive — c'est cette énumération, refaite de zéro, qui fait foi.

**Chiffres re-dérivés, jamais recopiés** : les chiffres structurels de l'espace de Robinson-Goforth
(75 ordres faibles, 5625 jeux à rangs, 576 chambres) sont **re-dérivés** dans chaque section ci-dessous
depuis zéro, pur stdlib — rien n'est recopié de GT-3b.

## 0. Configuration

Représentation héritée de GT-3b / GT-21 : un jeu ordinal 2×2 = une paire de tables de rangs
`(ligne, colonne)`, chacune un 4-uplet de rangs `{1..4}` lu en ligne-major — `t[0]` et `t[1]` sont la
première ligne, `t[2]` et `t[3]` la seconde. Le rang 4 est le meilleur payoff. Tout est **pur stdlib** :
l'univers tient en mémoire, aucune dépendance n'est nécessaire pour énumérer un espace fini.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Globalization;

// === Configuration : GameTheory 20, chemin minimal - temoin construit et verifie independamment ===
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture; // affichage numerique deterministe
Console.WriteLine("GameTheory 20 : chemin minimal (temoin construit et verifie independamment)");
Console.WriteLine("Representation : jeu = (table_ligne, table_colonne), rangs {1..4}, 4 = meilleur");
Console.WriteLine("Loi II : le constructeur PRODUIT le chemin, le verificateur sepere le VERIFIE independamment");


The below script needs to be able to find the current output cell; this is an easy method to get it.

GameTheory 20 : chemin minimal (temoin construit et verifie independamment)


Representation : jeu = (table_ligne, table_colonne), rangs {1..4}, 4 = meilleur


Loi II : le constructeur PRODUIT le chemin, le verificateur sepere le VERIFIE independamment


**Lecture de la sortie committée** : la sortie se contente de trois lignes — les partis pris du
carnet, sans encore aucun dénombrement. D'abord la **représentation** : « jeu = (table_ligne,
table_colonne), rangs {1..4}, 4 = meilleur » — un jeu est une paire de tables de rangs, le rang
4 portant le meilleur payoff. Puis la **Loi II** : le constructeur **PRODUIT** le chemin,
et le **vérificateur séparé** le **vérifie indépendamment**. C'est ce double mouvement — produire le
chemin, puis le faire vérifier par un organe séparé — que les sections 3 et 4 soumettent à
l'épreuve sur l'univers fini des jeux.

## 1. L'univers fini, re-dérivé depuis zéro

Un **ordre faible** sur 4 cases = un classement avec ex æquo possibles. La clé canonique relabelle
les valeurs distinctes par ordre de première apparition : deux 4-uplets décrivant le même classement
partagent la même clé. Le nombre d'ordres faibles sur 4 éléments est le **nombre de Fubini** F₄ ;
le notebook le mesure au lieu de le réciter.

In [2]:
// === Section 1.1 : enumeration des ordres faibles sur 4 cases ===

static string Canonique(string t) {
    // Cle canonique d'un 4-uplet : relabelage croissant des valeurs distinctes.
    var vals = t.Distinct().OrderBy(c => c).ToArray();
    var sb = new StringBuilder(4);
    foreach (var c in t) sb.Append((char)('1' + Array.IndexOf(vals, c)));
    return sb.ToString();
}

var tous = new List<string>();
for (int a = 1; a <= 4; a++)
for (int b = 1; b <= 4; b++)
for (int c = 1; c <= 4; c++)
for (int d = 1; d <= 4; d++)
    tous.Add(Canonique("" + (char)('0' + a) + (char)('0' + b) + (char)('0' + c) + (char)('0' + d)));
var ordresFaibles = tous.Distinct().OrderBy(x => x).ToList();
var stricts = ordresFaibles.Where(t => t.Distinct().Count() == 4).ToList();
var avecExAequo = ordresFaibles.Where(t => t.Distinct().Count() < 4).ToList();
Console.WriteLine("Ordres faibles sur 4 cases : " + ordresFaibles.Count + " (nombre de Fubini F4)");
Console.WriteLine("  stricts (futurs chambers)  : " + stricts.Count);
Console.WriteLine("  avec ex aequo (futurs murs) : " + avecExAequo.Count);
Console.WriteLine("Univers des jeux a rangs :  " + ordresFaibles.Count + " x " + ordresFaibles.Count + " = " + ordresFaibles.Count * ordresFaibles.Count);
Console.WriteLine("Univers des chambres strictes : " + stricts.Count * stricts.Count);


Ordres faibles sur 4 cases : 75 (nombre de Fubini F4)


  stricts (futurs chambers)  : 24


  avec ex aequo (futurs murs) : 51


Univers des jeux a rangs :  75 x 75 = 5625


Univers des chambres strictes : 576


**Lecture de la sortie committée** : **75** ordres faibles — c'est bien le nombre de Fubini F₄,
re-dérivé et non récité. Ils se partitionnent en **24 stricts** et **51 avec ex æquo**, d'où :
**5625 jeux à rangs** (l'univers complet, murs compris) et **576 chambres strictes** (l'univers de
Robinson-Goforth classique). Ces trois chiffres étaient RAPPORTÉS depuis la digestion ; ils sont
désormais VÉRIFIÉS par ce notebook, indépendamment de GT-3b.

### 1.2 Les jeux canoniques, encodés

Trois jeux classiques servent de bornes concrètes au constructeur. Encodage standard : pour un jeu
symétrique de dilemme 2×2 avec payoffs ordinaux `T > R > P > S` (défection tentante, punition mutuelle),
la table ligne lit `(R, S, T, P)` et la table colonne `(R, T, S, P)` — la transposition reflète le fait
que chaque joueur lit la même grille de son propre côté. Le Dilemme du Prisonnier suit `T > R > P > S`,
la Poule (Chicken) `T > R > S > P`, la Chasse au Cerf `R > T > P > S`.

In [3]:
// === Section 1.2 : les jeux canoniques (convention GT-21 / GT-3b) ===

static Tuple<string, string> Jeu(string row, string col) => Tuple.Create(row, col);

static void AfficherJeu(string nom, string row, string col) {
    Console.WriteLine(nom);
    Console.WriteLine($"  Ligne    | {row[0].ToString().PadLeft(2)}  {row[1].ToString().PadRight(2)}|   Colonne | {col[0].ToString().PadLeft(2)}  {col[1].ToString().PadRight(2)}|");
    Console.WriteLine($"           | {row[2].ToString().PadLeft(2)}  {row[3].ToString().PadRight(2)}|           | {col[2].ToString().PadLeft(2)}  {col[3].ToString().PadRight(2)}|");
}

var PD       = Jeu("3142", "3412");   // T>R>P>S  (encodage GT-3b)
var POULE    = Jeu("3241", "3421");   // T>R>S>P
var CERF     = Jeu("4132", "4312");   // R>T>P>S
var IDENTITE = Jeu("1234", "1234");
var RENVERSE = Jeu("4321", "4321");

foreach (var (nom, j) in new[] {
    ("Dilemme du Prisonnier", PD), ("Poule (Chicken)", POULE),
    ("Chasse au Cerf", CERF), ("Identite", IDENTITE), ("Renversement", RENVERSE) }) {
    AfficherJeu(nom, j.Item1, j.Item2);
    bool strict = j.Item1.Distinct().Count() == 4 && j.Item2.Distinct().Count() == 4;
    Console.WriteLine("  strict (chambre) : " + strict);
}
Console.WriteLine();
Console.WriteLine("Les cinq bornes canoniques sont des chambres strictes, toutes dans les 576.");


Dilemme du Prisonnier


  Ligne    |  3  1 |   Colonne |  3  4 |


           |  4  2 |           |  1  2 |


  strict (chambre) : True


Poule (Chicken)


  Ligne    |  3  2 |   Colonne |  3  4 |


... (11 outputs left out)

  Ligne    |  4  3 |   Colonne |  4  3 |


           |  2  1 |           |  2  1 |


  strict (chambre) : True


Les cinq bornes canoniques sont des chambres strictes, toutes dans les 576.


**Lecture de la sortie committée** : la sortie imprime les cinq jeux canoniques, chacun comme
une paire de tables de rangs — **Dilemme du Prisonnier** (Ligne 3 1 / 4 2, Colonne 3 4 / 1 2),
**Poule (Chicken)**, **Chasse au Cerf**, **Identité**, **Renversement** — et chaque table
emploie les quatre rangs {1, 2, 3, 4} exactement une fois, ce que récapitule le verdict «
strict (chambre) : True » répété cinq fois. Aux deux extrêmes, **Identité** répète la table 1 2
/ 3 4 des deux côtés et **Renversement** la table renversée 4 3 / 2 1 : les deux bornes sont
une permutation et son renversement. La dernière ligne conclut : « Les cinq bornes canoniques
sont des chambres strictes, toutes dans les 576. » — les cinq points de repère du cours sont
donc des sommets de l'univers strict dénombré à la section 1, et c'est entre eux que le
constructeur devra produire des chemins.

## 2. Les six swaps générateurs : le permutaèdre des chambres

Sur les chambres, la transformation élémentaire de Robinson-Goforth est l'**échange des valeurs
adjacentes k et k+1** dans une table : on permute les *cases* portant ces deux rangs. Trois échanges
sont possibles par table (k = 1, 2, 3), donc **six par jeu** — ce sont les « six swaps générateurs »
de la digestion. Le graphe obtenu sur les 24 ordres d'un côté est le **permutaèdre** S₄ ; sur les
576 chambres, son carré.

In [4]:
// === Section 2.1 : generateur de swap et adjacences des chambres ===

static string SwapValeursAdjacentes(string t, int k) {
    // Echange les cases portant les valeurs k et k+1 (traversee de la facette k<->k+1).
    int pk = t.IndexOf((char)('0' + k));
    int pk1 = t.IndexOf((char)('0' + (k + 1)));
    var l = t.ToCharArray();
    (l[pk], l[pk1]) = (l[pk1], l[pk]);
    return new string(l);
}

static Tuple<string, string> SwapJeu(Tuple<string, string> jeu, string cote, int k) {
    var row = jeu.Item1; var col = jeu.Item2;
    if (cote == "ligne") return Tuple.Create(SwapValeursAdjacentes(row, k), col);
    return Tuple.Create(row, SwapValeursAdjacentes(col, k));
}

// Adjacences sur les 576 chambres : 6 voisins par jeu
var chambres = new List<Tuple<string, string>>();
foreach (var r in stricts) foreach (var c in stricts) chambres.Add(Tuple.Create(r, c));
var adjChambres = new Dictionary<Tuple<string, string>, List<Tuple<string, string>>>();
foreach (var g in chambres) {
    var v = new List<Tuple<string, string>>();
    foreach (var cote in new[] { "ligne", "colonne" })
        for (int k = 1; k <= 3; k++) v.Add(SwapJeu(g, cote, k));
    adjChambres[g] = v;
}
Console.WriteLine("Chambres : " + chambres.Count + " | degre de chaque sommet : " + adjChambres[chambres[0]].Count);

// BFS exhaustif depuis l'identite : connexite, diametre, distribution des distances
static Dictionary<T, int> BfsComplet<T>(T depart, Func<T, List<T>> voisins) {
    var dist = new Dictionary<T, int> { { depart, 0 } };
    var q = new Queue<T>();
    q.Enqueue(depart);
    while (q.Count > 0) {
        var u = q.Dequeue();
        foreach (var v in voisins(u))
            if (!dist.ContainsKey(v)) { dist[v] = dist[u] + 1; q.Enqueue(v); }
    }
    return dist;
}

var distIdentite = BfsComplet(IDENTITE, g => adjChambres[g]);
Console.WriteLine("Graphe des chambres : connexe = " + (distIdentite.Count == 576) + " | diametre = " + distIdentite.Values.Max());
Console.WriteLine("  repartition des distances : " + string.Join(", ", distIdentite.Values.GroupBy(v => v).OrderBy(x => x.Key).Select(g => "(" + g.Key + ", " + g.Count() + ")")));


Chambres : 576 | degre de chaque sommet : 6


Graphe des chambres : connexe = True | diametre = 12


  repartition des distances : (0, 1), (1, 6), (2, 19), (3, 42), (4, 71), (5, 96), (6, 106), (7, 96), (8, 71), (9, 42), (10, 19), (11, 6), (12, 1)


**Lecture de la sortie committée** : le graphe des 576 chambres est **connexe**, de diamètre **12** —
six par côté, la distance d'un jeu étant la somme des distances de ses deux tables. La distribution
est **symétrique** (même effectif à d et 12−d) : l'involution qui renverse tous les rangs des deux
tables est une isométrie du permutaèdre. Le diamètre par table seule est 6, atteint par le renversement
`(1,2,3,4) → (4,3,2,1)` : chacune des six paires de positions doit être corrigée exactement une fois.

## 3. Le constructeur

C'est le geste que le vérificateur ne sait pas faire : étant donnés deux jeux **G** et **H**, **produire**
une suite d'échanges élémentaires qui mène de l'un à l'autre. Le constructeur parcourt le permutaèdre
en largeur depuis G, mémorise pour chaque sommet visité le move qui y a mené, puis **remonter la
chaîne des parents** depuis H exhume le chemin. Il ne connaît ni les jeux classiques, ni les noms —
il ne connaît que le graphe.

In [5]:
// === Section 3.1 : construire_chemin -- le constructeur temoin ===

static List<Tuple<T, T>> ConstruireChemin<T>(T depart, T arrivee, Func<T, List<T>> voisins) {
    // Produit une suite (jeu, move) de depart a arrivee par BFS + remontee des parents.
    var parent = new Dictionary<T, T>();
    var visited = new HashSet<T> { depart };
    var q = new Queue<T>();
    q.Enqueue(depart);
    while (q.Count > 0) {
        var u = q.Dequeue();
        if (u.Equals(arrivee)) break;
        foreach (var v in voisins(u))
            if (visited.Add(v)) { parent[v] = u; q.Enqueue(v); }
    }
    if (!visited.Contains(arrivee)) return null;
    var chaine = new List<Tuple<T, T>>();
    var cur = arrivee;
    while (!cur.Equals(depart)) { var pred = parent[cur]; chaine.Add(Tuple.Create(pred, cur)); cur = pred; }
    chaine.Reverse();
    return chaine;
}

static string NommerPas(Tuple<string, string> pred, Tuple<string, string> cur) {
    // Nomme le move elementaire entre deux jeux consecutifs (cote, valeurs echangees).
    var cotes = new[] { ("ligne", 0), ("colonne", 1) };
    foreach (var (cote, table) in cotes) {
        var pt = table == 0 ? pred.Item1 : pred.Item2;
        var ct = table == 0 ? cur.Item1 : cur.Item2;
        if (pt != ct)
            for (int k = 1; k <= 3; k++)
                if (SwapJeu(pred, cote, k).Equals(cur))
                    return $"swap {cote} ({k} <-> {k + 1})";
    }
    return "?";
}

var cheminPdPoule = ConstruireChemin(PD, POULE, g => adjChambres[g]);
Console.WriteLine("Chemin minimal construit : Dilemme du Prisonnier -> Poule");
Console.WriteLine($"  {PD.Item1} | {PD.Item2}   (depart)");
foreach (var pc in cheminPdPoule)
    Console.WriteLine($"  -- {NommerPas(pc.Item1, pc.Item2)} -->  {pc.Item2.Item1} | {pc.Item2.Item2}");
Console.WriteLine($"Longueur du chemin produit : {cheminPdPoule.Count} pas");


Chemin minimal construit : Dilemme du Prisonnier -> Poule


  3142 | 3412   (depart)


  -- swap ligne (1 <-> 2) -->  3241 | 3412


  -- swap colonne (1 <-> 2) -->  3241 | 3421


Longueur du chemin produit : 2 pas


**Lecture de la sortie committée** : le constructeur **produit** le chemin — deux pas exactement :
un échange `(1 ↔ 2)` côté ligne (la punition mutuelle P et la soumission S échangent leurs rangs) et
un échange `(1 ↔ 2)` côté colonne. Géométriquement, PD et Poule sont deux chambres voisines-à-deux-
pas dans le permutaèdre : ils ne diffèrent que par l'ordre des deux **mauvais** payoffs, le meilleur
(T) et le moyen (R) étant aux mêmes positions. Le dilemme et le jeu du poule : deux mondes séparés
par la seule question « quelle misère préférer ». La machine ne le savait pas — elle l'a construit.

In [6]:
// === Section 3.2 : le constructeur sur l'antipode ===

var cheminRenverse = ConstruireChemin(IDENTITE, RENVERSE, g => adjChambres[g]);
Console.WriteLine("Chemin minimal construit : Identite -> Renversement (les deux antipodes du permutaedre)");
for (int i = 0; i < cheminRenverse.Count; i++)
    Console.WriteLine($"  pas {(i + 1).ToString().PadLeft(2)} : {NommerPas(cheminRenverse[i].Item1, cheminRenverse[i].Item2)}");
Console.WriteLine($"Longueur : {cheminRenverse.Count} pas (le diametre exact du permutaedre S4 x S4)");

var cheminPdCerf = ConstruireChemin(PD, CERF, g => adjChambres[g]);
Console.WriteLine();
var mvList = string.Join(", ", cheminPdCerf.Select(pc => NommerPas(pc.Item1, pc.Item2)));
Console.WriteLine($"Dilemme -> Chasse au Cerf : {cheminPdCerf.Count} pas : [{mvList}]");


Chemin minimal construit : Identite -> Renversement (les deux antipodes du permutaedre)


  pas  1 : swap ligne (1 <-> 2)


  pas  2 : swap ligne (2 <-> 3)


  pas  3 : swap ligne (1 <-> 2)


  pas  4 : swap ligne (3 <-> 4)


  pas  5 : swap ligne (2 <-> 3)


... (5 outputs left out)

  pas 11 : swap colonne (2 <-> 3)


  pas 12 : swap colonne (1 <-> 2)


Longueur : 12 pas (le diametre exact du permutaedre S4 x S4)


Dilemme -> Chasse au Cerf : 2 pas : [swap ligne (3 <-> 4), swap colonne (3 <-> 4)]


**Lecture de la sortie committée** : l'antipode exige les **12 pas** du diamètre — chaque paire de positions des deux tables doit être corrigée exactement une fois, et le constructeur les exhume toutes. Ces douze pas se répartissent en deux blocs de six — swaps ligne aux pas 1 à 6, puis swaps colonne aux pas 7 à 12 : chaque table est renversée indépendamment, image directe de la structure produit S₄ × S₄. PD → Cerf est plus proche qu'il n'y paraît : deux pas, les échanges `(3 ↔ 4)` de chaque côté — seul l'ordre du **meilleur** payoff (coopération mutuelle R contre défection tentante T) change. La paire dilemme/cerf, structuralement opposée en théorie des jeux (dilemme social contre jeu de coordination), est géométriquement contiguë : c'est la lecture de Robinson-Goforth, reconstruite par la machine.


## 4. Le vérificateur séparé — le témoin

Produire un chemin ne suffit pas : encore faut-il le **vérifier indépendamment** — validité **et**
minimalité. Le témoin honnête exige un vérificateur qui ne réutilise **aucun état** du constructeur : il
re-dérive tout. Trois verdicts doivent être distingués — sinon le vérificateur n'est qu'un tampon :

- **VALIDE + MINIMAL** : chaque pas est un swap élémentaire, les extrémités sont exactes, et la
  longueur égale la distance exhaustive d(G, H) ;
- **VALIDE + NON MINIMAL** : le chemin marche mais un détour existe ;
- **INVALIDE** : au moins un pas n'est pas un move élémentaire (ou les extrémités mentent).

La minimalité se prouve par **recherche exhaustive** : BFS complet depuis G — l'univers est fini,
5625 ou 576 sommets, l'énumération totale EST la preuve qu'aucun raccourci n'existe.

In [7]:
// === Section 4.1 : verifier_chemin -- le verificateur independant ===

static bool PasElementaireValide(Tuple<string, string> pred, Tuple<string, string> cur, Dictionary<Tuple<string, string>, List<Tuple<string, string>>> adj) {
    // True ssi cur s'obtient de pred par exactement un swap elementaire.
    return adj.ContainsKey(pred) && adj[pred].Contains(cur);
}

static string VerifierChemin(Tuple<string, string> depart, Tuple<string, string> arrivee,
    List<Tuple<Tuple<string, string>, Tuple<string, string>>> chaine,
    Dictionary<Tuple<string, string>, List<Tuple<string, string>>> adj,
    Func<Tuple<string, string>, List<Tuple<string, string>>> voisins) {
    // Verdict independant : re-derive TOUT, ne reutilise rien du constructeur.
    if (chaine == null || chaine.Count == 0) return "INVALIDE : chemin vide";
    if (!chaine[0].Item1.Equals(depart) || !chaine[chaine.Count - 1].Item2.Equals(arrivee))
        return "INVALIDE : les extremites ne sont pas celles annoncees";
    foreach (var pc in chaine)
        if (!PasElementaireValide(pc.Item1, pc.Item2, adj))
            return $"INVALIDE : pas non elementaire vers {pc.Item2}";
    var dist = BfsComplet(depart, voisins);   // recherche exhaustive : la preuve de minimalite
    int dReel = dist[arrivee];
    if (chaine.Count != dReel) return $"VALIDE mais NON MINIMAL : {chaine.Count} pas, distance reelle {dReel}";
    return $"VALIDE + MINIMAL : {chaine.Count} pas = distance exhaustive d(G,H)";
}

// Les trois verdicts, demontres sur des cas construits pour eux
Console.WriteLine("1. Le chemin du constructeur (PD -> Poule) :");
Console.WriteLine("   " + VerifierChemin(PD, POULE, cheminPdPoule, adjChambres, g => adjChambres[g]));

var voisinPoule = SwapJeu(POULE, "ligne", 1);   // un voisin quelconque de l'arrivee
var cheminDetour = new List<Tuple<Tuple<string, string>, Tuple<string, string>>>(cheminPdPoule);
cheminDetour.Add(Tuple.Create(POULE, voisinPoule));
cheminDetour.Add(Tuple.Create(voisinPoule, POULE));
Console.WriteLine("2. Le meme chemin rallonge d'un aller-retour :");
Console.WriteLine("   " + VerifierChemin(PD, POULE, cheminDetour, adjChambres, g => adjChambres[g]));

var fauxPas = new List<Tuple<Tuple<string, string>, Tuple<string, string>>> { Tuple.Create(PD, CERF) };
Console.WriteLine("3. Un 'chemin' en un pas vers un jeu non adjacent :");
Console.WriteLine("   " + VerifierChemin(PD, CERF, fauxPas, adjChambres, g => adjChambres[g]));


1. Le chemin du constructeur (PD -> Poule) :


   VALIDE + MINIMAL : 2 pas = distance exhaustive d(G,H)


2. Le meme chemin rallonge d'un aller-retour :


   VALIDE mais NON MINIMAL : 4 pas, distance reelle 2


3. Un 'chemin' en un pas vers un jeu non adjacent :


   INVALIDE : pas non elementaire vers (4132, 4312)


**Lecture de la sortie committée** : les trois verdicts tombent, distincts. Le chemin du constructeur
est **VALIDE + MINIMAL** (2 pas = distance exhaustive). Le même chemin avec un aller-retour greffé est
**VALIDE mais NON MINIMAL** — le vérificateur voit le détour. Et le saut direct PD → Cerf est
**INVALIDE** : un pas non élémentaire, parce que les deux chambres ne sont pas adjacentes. Le
vérificateur n'est pas un tampon : il rend trois réponses différentes sur trois défauts différents.
C'est la séparation constructeur/vérificateur de la Loi II — le même geste que la preuve `by decide`
du translateur de Life, ici sur le substrat fini des jeux.

### 4.2 La non-unicité : compter les géodésiques

Un chemin minimal n'est **jamais seul**. Les géodésiques entre deux jeux forment les « mondes
minimaux » — autant de façons distinctes de transformer l'un en l'autre sans un pas de trop. Le
comptage se fait par le DAG des couches BFS : un sommet de couche d reçoit ses chemins de ses
prédécesseurs de couche d−1. Pour le renversement d'une table (l'antipode du permutaèdre S₄),
la combinatoire des groupes de Coxeter prédit que le nombre de mots réduits de l'élément le plus
long w₀ de S₄ vaut 16 — une prédiction externe que la mesure peut contredire.

In [8]:
// === Section 4.2 : comptage des geodesiques par le DAG des couches ===

static int CompterGeodesiques<T>(T depart, T arrivee, Func<T, List<T>> voisins) {
    // Nombre de chemins minimaux : produits des chemins entrants, couche par couche.
    var dist = BfsComplet(depart, voisins);
    int d = dist[arrivee];
    var nb = new Dictionary<T, int> { { depart, 1 } };
    for (int couche = 1; couche <= d; couche++)
        foreach (var kvp in dist)
            if (kvp.Value == couche)
                nb[kvp.Key] = voisins(kvp.Key).Where(v => dist.TryGetValue(v, out var dv) && dv == couche - 1).Sum(v => nb[v]);
    return nb[arrivee];
}

// Cote seul : 24 ordres, antipode (1,2,3,4) -> (4,3,2,1)
var adjTable = new Dictionary<string, List<string>>();
foreach (var t in stricts) {
    var v = new List<string>();
    for (int k = 1; k <= 3; k++) v.Add(SwapValeursAdjacentes(t, k));
    adjTable[t] = v;
}
int geosTable = CompterGeodesiques("1234", "4321", t => adjTable[t]);
Console.WriteLine($"Geodesiques (1,2,3,4) -> (4,3,2,1) sur une table : {geosTable} (mots reduits de w0 dans S4 : 16 attendus)");

// Jeu complet : PD -> Poule
int geosPdPoule = CompterGeodesiques(PD, POULE, g => adjChambres[g]);
Console.WriteLine($"Geodesiques PD -> Poule sur les 576 chambres : {geosPdPoule}");

int geosRenverse = CompterGeodesiques(IDENTITE, RENVERSE, g => adjChambres[g]);
Console.WriteLine($"Geodesiques Identite -> Renversement (diametre 12) : {geosRenverse}");


Geodesiques (1,2,3,4) -> (4,3,2,1) sur une table : 16 (mots reduits de w0 dans S4 : 16 attendus)


Geodesiques PD -> Poule sur les 576 chambres : 2


Geodesiques Identite -> Renversement (diametre 12) : 236544


**Lecture de la sortie committée** : la mesure donne **16** géodésiques pour le renversement d'une
table — exactement le nombre de mots réduits de w₀ dans S₄ que prédit la théorie des groupes de
Coxeter. La mesure rejoint la combinatoire connue : le DAG des couches ne calcule rien d'autre que
les expressions réduites du plus long élément. Pour le jeu complet, la composition n'est **pas** un
simple produit : les pas des deux côtés s'**entrelacent**. PD → Poule (un pas par table) offre
**2** géodésiques — l'ordre des deux pas ; l'antipode complet en offre **236 544 = 16 × 16 × 924**,
les mots réduits par côté multipliés par les C(12,6) = 924 entrelacements des deux séquences de six
pas. Le chemin produit au §3 n'était qu'un tirage parmi cette flore.

## 5. L'univers à rangs complet : traverser les murs

Le permutaèdre ne couvre que les chambres. L'univers à rangs complet (5625) inclut les **murs** —
les jeux à égalités, strates de codimension ≥ 1 de GT-3b. Deux moves élémentaires s'ajoutent aux
swaps : **make_tie** (coalescence des valeurs k et k+1 : la chambre descend sur un mur) et son
inverse **break_tie** (résoudre l'ex æquo dans un sens ou l'autre : traverser). En graphe non
orienté, les deux directions d'un même edge suffisent : coalescer depuis chacune des chambres
adjacentes énumère toutes les résolutions du mur.

La question que seul l'univers complet peut trancher : **les murs sont-ils des raccourcis ?**
Un chemin qui passe par un mur peut-il être plus court que la géodésique des chambres ? L'intuition
suggère que oui — un swap dans un mur déplace un bloc fusionné d'un seul coup. La mesure va
décider, et elle ne suit pas l'intuition.

In [9]:
// === Section 5.1 : l'univers complet, swaps + make_tie, et son diametre ===

static string FaireLeTie(string t, int k) {
    // Coalescence des valeurs k et k+1 -> forme canonique du mur.
    var sb = new StringBuilder(t.Length);
    foreach (var c in t) {
        int v = c - '0';
        sb.Append((char)('0' + (v == k + 1 ? k : v)));
    }
    return Canonique(sb.ToString());
}

static List<Tuple<string, string>> VoisinsComplets(Tuple<string, string> jeu) {
    // Swaps (toujours valides dans une table a rangs) + make_tie (descente sur un mur).
    // Break_tie = le meme edge pris dans l'autre sens.
    var row = jeu.Item1; var col = jeu.Item2;
    var res = new List<Tuple<string, string>>();
    var cotes = new[] { ("ligne", row), ("colonne", col) };
    foreach (var (cote, t) in cotes) {
        var valeurs = new HashSet<char>(t);
        for (int k = 1; k <= 3; k++)
            if (valeurs.Contains((char)('0' + k)) && valeurs.Contains((char)('0' + (k + 1)))) {
                res.Add(SwapJeu(jeu, cote, k));
                var t2 = FaireLeTie(t, k);
                if (t2 != t) res.Add(cote == "ligne" ? Tuple.Create(t2, col) : Tuple.Create(row, t2));
            }
    }
    return res;
}

var univers = new List<Tuple<string, string>>();
foreach (var r in ordresFaibles) foreach (var c in ordresFaibles) univers.Add(Tuple.Create(r, c));
var distUnivers = BfsComplet(IDENTITE, VoisinsComplets);
Console.WriteLine($"Univers complet : {univers.Count} sommets, atteints : {distUnivers.Count} (connexe : {distUnivers.Count == univers.Count})");
Console.WriteLine($"Diametre de l'univers complet : {distUnivers.Values.Max()}");
Console.WriteLine("  (a comparer au diametre 12 du seul permutaedre des chambres)");
Console.WriteLine("  repartition : " + string.Join(", ", distUnivers.Values.GroupBy(v => v).OrderBy(x => x.Key).Select(g => "(" + g.Key + ", " + g.Count() + ")")));


Univers complet : 5625 sommets, atteints : 5625 (connexe : True)


Diametre de l'univers complet : 12


  (a comparer au diametre 12 du seul permutaedre des chambres)


  repartition : (0, 1), (1, 12), (2, 68), (3, 240), (4, 578), (5, 990), (6, 1232), (7, 1128), (8, 785), (9, 402), (10, 149), (11, 36), (12, 4)


**Lecture de la sortie committée** : l'univers complet est **connexe** — coalescences et traversées relient les 5625 — mais son diamètre **reste 12**, identique au permutaèdre des chambres, avec les mêmes 4 antipodes. La répartition culmine à 1232 sommets à la distance 6 — le milieu exact de l'intervalle [0, 12] — la distribution garde sa masse maximale en couche 6 et retombe exactement sur les mêmes extrêmes. L'intuition « les murs raccourcissent » vient de se heurter à un premier fait : au niveau du diamètre, rien ne bouge. Reste à savoir si les murs raccourcissent **certaines** distances locales — c'est l'objet du §5.2, qui le décide par énumération exhaustive.


In [10]:
// === Section 5.2 : les murs raccourcissent-ils ? decision par enumeration exhaustive ===

// Les generateurs ne touchent qu'un cote a la fois : les deux graphes sont des produits.
// Il suffit de comparer, cote par cote, les distances des 24 x 24 paires d'ordres stricts
// dans le permutaedre (24 noeuds) et dans l'espace a rangs complet (75 noeuds).

static List<string> VoisinsCotePerm(string t) {
    var res = new List<string>();
    for (int k = 1; k <= 3; k++)
        if (t.IndexOf((char)('0' + k)) >= 0 && t.IndexOf((char)('0' + (k + 1))) >= 0)
            res.Add(SwapValeursAdjacentes(t, k));
    return res;
}

static List<string> VoisinsCoteFull(string t) {
    var res = new List<string>();
    for (int k = 1; k <= 3; k++)
        if (t.IndexOf((char)('0' + k)) >= 0 && t.IndexOf((char)('0' + (k + 1))) >= 0) {
            res.Add(SwapValeursAdjacentes(t, k));
            var t2 = FaireLeTie(t, k);
            if (t2 != t) res.Add(t2);
        }
    return res;
}

int ecarts = 0;
foreach (var a in stricts) {
    var dPerm = BfsComplet(a, VoisinsCotePerm);
    var dFull = BfsComplet(a, VoisinsCoteFull);
    foreach (var b in stricts)
        if (dPerm[b] != dFull[b]) ecarts++;
}
Console.WriteLine("Paires d'ordres stricts comparees (24 x 24) : 576");
Console.WriteLine($"Paires ou l'espace complet (murs autorises) est PLUS COURT que le permutaedre : {ecarts}");
Console.WriteLine();
Console.WriteLine($"Par structure produit, le verdict couvre les {chambres.Count} x {chambres.Count} = {chambres.Count * chambres.Count} paires de chambres :");
Console.WriteLine("  les murs ne raccourcissent AUCUNE distance inter-chambres.");

// Le constructeur, confirme sur un cas : il ignore les murs de lui-meme
var cheminFull = ConstruireChemin(PD, RENVERSE, VoisinsComplets);
int mursTraverses = 0;
foreach (var pc in cheminFull)
    if (pc.Item2.Item1.Distinct().Count() < 4 || pc.Item2.Item2.Distinct().Count() < 4) mursTraverses++;
var cheminPur = ConstruireChemin(PD, RENVERSE, g => adjChambres[g]);
Console.WriteLine();
Console.WriteLine($"PD -> Renversement : univers complet {cheminFull.Count} pas dont {mursTraverses} sur un mur ; chambres seules {cheminPur.Count} pas");
Console.WriteLine("Verificateur (univers complet) : " + VerifierChemin(PD, RENVERSE, cheminFull, adjChambres, VoisinsComplets));


Paires d'ordres stricts comparees (24 x 24) : 576


Paires ou l'espace complet (murs autorises) est PLUS COURT que le permutaedre : 0


Par structure produit, le verdict couvre les 576 x 576 = 331776 paires de chambres :


  les murs ne raccourcissent AUCUNE distance inter-chambres.


PD -> Renversement : univers complet 5 pas dont 0 sur un mur ; chambres seules 5 pas


Verificateur (univers complet) : VALIDE + MINIMAL : 5 pas = distance exhaustive d(G,H)


**Lecture de la sortie committée** : la décision est **nulle et exhaustive** — sur les 576 paires
d'ordres stricts comparées côté par côté, **aucune** n'est plus courte par les murs ; et parce que
les générateurs ne touchent qu'un côté à la fois (structure produit), ce verdict par côté couvre
les 331 776 paires de chambres sans exception. Le mécanisme se voit sur le cas concret : le
constructeur lâché dans l'univers complet produit un chemin qui **ne pose jamais le pied sur un
mur** — la même longueur, 5 pas, que dans les chambres seules, et le vérificateur séparé l'atteste
minimal dans l'univers élargi. Descendre sur un mur coûte un pas, en sortir en coûte un autre,
et le swap qui déplace le bloc fusionné ne récupère jamais plus que ces deux pas investis.
Les murs de GT-3b ne sont donc pas des tunnels : ce sont des strates que la géométrie des chambres
regarde traverser sans jamais y gagner un pas. Un résultat négatif — mais prouvé par énumération
complète, et c'est lui qui donne sa valeur au témoin : la minimalité ne dit pas seulement
« pas de détour », elle dit « pas de détour, même par les murs ».

## 6. L'epaisseur des murs — les chambres en forme extensive

Les cinq sections precedentes repondent a une question de **classement** : quelle chambre contient
cette matrice, et quel est le chemin minimal d'une chambre a l'autre. En forme normale, le mur entre
deux chambres est une frontiere de nomenclature : on le franchit en **changeant de matrice**, pas en
jouant. La question extensive deplace le sujet : *quel chemin causal et informationnel rend une
issue atteignable a matrice fixee ?* Le mur cesse d'etre une etiquette pour devenir un objet
**traversable-sous-conditions** — ce qui le franchit est une promesse ou une menace **croyable**,
et la croyabilite est exactement ce que l'arbre sait calculer (sous-jeux, induction a rebours) ;
le mur `w = l` y est un separateur de codimension 1, pas un nom.

**Le dispositif.** Avant le jeu de base, le joueur Ligne choisit un **engagement observable** :
se taire, promettre (se lier a Cooperer — la force acquise en se privant d'une option, Schelling),
ou menacer (programmer une punition si Colonne defectionne). Chaque variante devient un arbre
explicite : noeuds de decision, ensembles d'information (singletons ici, l'engagement etant
observe — la partition est imprimee avec l'arbre), feuilles valorisees. La croyabilite n'est
jamais assertee : elle se **decide par induction a rebours au sous-jeu de punition**.

**Unites.** Les issues du jeu de base gardent les rangs {1..4} de la chambre ; l'issue de punition
porte des utilites cardinales (0,0), sous le plancher ordinal des rangs — n'importe quelle
utilite coherente avec l'ordre de la chambre donne le meme verdict (robustesse ordinale).

In [11]:
// === Section 6.1 : moteur de forme extensive -- arbre, ensembles d'information, induction a rebours ===

class Noeud {
    public string Type;   // "feuille" | "decision"
    public int Joueur;
    public (int, int) U;  // utilites (u1, u2)
    public string Nom;
    public Dictionary<string, Noeud> Enfants;
    public string Info;   // etiquette de l'ensemble d'information
}

static Noeud Feuille(int u1, int u2, string nom) =>
    new Noeud { Type = "feuille", U = (u1, u2), Nom = nom };

static Noeud Decision(int joueur, Dictionary<string, Noeud> enfants, string info) =>
    new Noeud { Type = "decision", Joueur = joueur, Enfants = enfants, Info = info };

static int Payoff((int, int) u, int joueur) => joueur == 0 ? u.Item1 : u.Item2;

static (int, int) ValeurSpe(Noeud noeud) {
    // Valeur d'equilibre parfait en sous-jeux par induction a rebours.
    if (noeud.Type == "feuille") return noeud.U;
    int joueur = noeud.Joueur;
    (int, int) best = (0, 0);
    bool first = true;
    foreach (var kv in noeud.Enfants) {
        var v = ValeurSpe(kv.Value);
        if (first || Payoff(v, joueur) > Payoff(best, joueur)) { best = v; first = false; }
    }
    return best;
}

static List<(List<string> trace, string nom, (int, int) u)> CheminSpe(Noeud noeud, List<string> trace = null) {
    // Decompte du chemin d'equilibre (actions choisies a chaque noeud).
    trace = trace ?? new List<string>();
    if (noeud.Type == "feuille")
        return new List<(List<string>, string, (int, int))> { (new List<string>(trace), noeud.Nom, noeud.U) };
    int joueur = noeud.Joueur;
    var bestU = ValeurSpe(noeud);
    var sorties = new List<(List<string>, string, (int, int))>();
    foreach (var kv in noeud.Enfants) {
        var v = ValeurSpe(kv.Value);
        if (Payoff(v, joueur) == Payoff(bestU, joueur)) {
            var newTrace = new List<string>(trace) { $"J{noeud.Joueur + 1}:{kv.Key}" };
            foreach (var s in CheminSpe(kv.Value, newTrace)) sorties.Add(s);
        }
    }
    return sorties;
}

static List<string> PartitionInformation(Noeud noeud, List<string> acc = null) {
    // Recense les ensembles d'information explicites de l'arbre.
    acc = acc ?? new List<string>();
    if (noeud.Type == "decision") {
        acc.Add(noeud.Info);
        foreach (var enfant in noeud.Enfants.Values) PartitionInformation(enfant, acc);
    }
    return acc;
}

Console.WriteLine("Moteur : induction a rebours sur arbre fini, ensembles d'information explicites.");
Console.WriteLine("Test de coherence sur un mini-arbre a menace choisie :");
// mini-arbre de controle : J1 punit ou laisse apres une defection observee
var test = Decision(0, new Dictionary<string, Noeud> {
    ["laisser"] = Feuille(1, 4, "laisser (S,T)"),
    ["punir"]   = Feuille(0, 0, "punir (0,0)"),
}, "test:punition");
var uTest = ValeurSpe(test);
var choix = CheminSpe(test)[0].trace[^1];
Console.WriteLine($"  sous-jeu de punition : valeur SPE = ({uTest.Item1}, {uTest.Item2}), action retenue = {choix}");
Console.WriteLine("  -> punir rapporte 0 < laisser 1 au punisseur : la menace CHOISIE est INCROYABLE");
Console.WriteLine("  (le verdict vient de l'induction, pas d'une assertion).");


Moteur : induction a rebours sur arbre fini, ensembles d'information explicites.


Test de coherence sur un mini-arbre a menace choisie :


  sous-jeu de punition : valeur SPE = (1, 4), action retenue = J1:laisser


  -> punir rapporte 0 < laisser 1 au punisseur : la menace CHOISIE est INCROYABLE


  (le verdict vient de l'induction, pas d'une assertion).


**Lecture de la sortie commitee** : sur le sous-jeu de punition seul, l'induction retient
*laisser* (1 > 0 pour le punisseur) — une menace **laissee au choix** de son auteur ne survit pas
a l'induction a rebours. C'est le critere operational de la section : la meme menace, **programmee**
(branche retiree de l'arbre), devient croyable par construction. Reste a mesurer ce que chaque
forme d'engagement fait au jeu de base, chambre par chambre.

In [12]:
// === Section 6.2 : le Dilemme en forme extensive -- trois engagements, trois arbres ===

static (int, int) ExtraireCellule(Tuple<string, string> jeu, int i, int j) {
    // (table_ligne, table_colonne) -> utilites cardinales (u1, u2) de la case (i,j)
    var tl = jeu.Item1; var tc = jeu.Item2;
    return (tl[2 * i + j] - '0', tc[2 * i + j] - '0');
}

var PD_CELLULES = new Dictionary<(int, int), (int, int)>();
foreach (var i in new[] { 0, 1 }) foreach (var j in new[] { 0, 1 }) PD_CELLULES[(i, j)] = ExtraireCellule(PD, i, j);
var CERF_CELLULES = new Dictionary<(int, int), (int, int)>();
foreach (var i in new[] { 0, 1 }) foreach (var j in new[] { 0, 1 }) CERF_CELLULES[(i, j)] = ExtraireCellule(CERF, i, j);
var P = PD_CELLULES;   // alias compact pour le Dilemme

static Noeud ArbreDilemme(string mode, Dictionary<(int, int), (int, int)> P) {
    // Arbre extensif du Dilemme selon l'engagement de Ligne (J0).
    // mode = 'silence'          : jeu de base simultane (aucune influence)
    // mode = 'promesse'         : J1 se lie a C (branche D retiree), engagement OBSERVABLE
    // mode = 'menace_choisie'   : apres (C,D), J1 choisit encore punir/laisser
    // mode = 'menace_programmee': apres (C,D), la punition est automatique (branche retiree)
    if (mode == "silence") {
        (int, int) suiteSilence(int j2) {
            int meilleureJ1 = P[(0, j2)].Item1 > P[(1, j2)].Item1 ? 0 : 1;
            return P[(meilleureJ1, j2)];
        }
        return Decision(1, new Dictionary<string, Noeud> {
            ["C"] = Feuille(suiteSilence(0).Item1, suiteSilence(0).Item2, "(BR,C)"),
            ["D"] = Feuille(suiteSilence(1).Item1, suiteSilence(1).Item2, "(BR,D)"),
        }, "silence:J2-jeu-de-base");
    }
    if (mode == "promesse")
        return Decision(1, new Dictionary<string, Noeud> {
            ["C"] = Feuille(P[(0, 0)].Item1, P[(0, 0)].Item2, "(C,C)"),
            ["D"] = Feuille(P[(0, 1)].Item1, P[(0, 1)].Item2, "(C,D)"),
        }, "promesse:J2-observe-C-force");
    if (mode == "menace_choisie") {
        var punition = Decision(0, new Dictionary<string, Noeud> {
            ["laisser"] = Feuille(P[(0, 1)].Item1, P[(0, 1)].Item2, "(C,D)-laissee"),
            ["punir"]   = Feuille(0, 0, "(C,D)-punie"),
        }, "menace:J1-choisit");
        return Decision(1, new Dictionary<string, Noeud> {
            ["C"] = Feuille(P[(0, 0)].Item1, P[(0, 0)].Item2, "(C,C)"),
            ["D"] = punition,
        }, "menace-choisie:J2-observe");
    }
    if (mode == "menace_programmee")
        return Decision(1, new Dictionary<string, Noeud> {
            ["C"] = Feuille(P[(0, 0)].Item1, P[(0, 0)].Item2, "(C,C)"),
            ["D"] = Feuille(0, 0, "(C,D)-punie-automatiquement"),
        }, "menace-programmee:J2-observe");
    throw new Exception("mode inconnu : " + mode);
}

var VERDICTS = new Dictionary<string, (int, int)>();
var sepInfoD = ", ";
var sepTraceD = " -> ";
foreach (var mode in new[] { "silence", "promesse", "menace_choisie", "menace_programmee" }) {
    var arbre = ArbreDilemme(mode, P);
    var u = ValeurSpe(arbre);
    var chemins = CheminSpe(arbre);
    var infos = PartitionInformation(arbre);
    VERDICTS[mode] = u;
    Console.WriteLine($"--- Dilemme / engagement = {mode}");
    Console.WriteLine($"    ensembles d'information : [{string.Join(sepInfoD, infos)}]");
    foreach (var (tr, nom, uu) in chemins)
        Console.WriteLine($"    chemin SPE {string.Join(sepTraceD, tr).PadRight(38)} issue {nom.PadRight(32)} utilites ({uu.Item1}, {uu.Item2})");
    Console.WriteLine($"    valeur SPE : ({u.Item1}, {u.Item2})");
}
Console.WriteLine();
var cooperative = P[(0, 0)];
Console.WriteLine($"Play cooperatif (C,C) = ({cooperative.Item1}, {cooperative.Item2}) (utilites du Dilemme)");
foreach (var kv in VERDICTS) {
    var u = kv.Value;
    var atteint = (u == cooperative) ? "ATTEINTE " : "inatteignable";
    Console.WriteLine($"  {kv.Key.PadRight(18)} -> ({u.Item1}, {u.Item2}) : transition vers le play cooperatif : {atteint}");
}


--- Dilemme / engagement = silence


    ensembles d'information : [silence:J2-jeu-de-base]


    chemin SPE J2:D                                   issue (BR,D)                           utilites (2, 2)


    valeur SPE : (2, 2)


--- Dilemme / engagement = promesse


    ensembles d'information : [promesse:J2-observe-C-force]


... (11 outputs left out)

  silence            -> (2, 2) : transition vers le play cooperatif : inatteignable


  promesse           -> (1, 4) : transition vers le play cooperatif : inatteignable


  menace_choisie     -> (1, 4) : transition vers le play cooperatif : inatteignable


  menace_programmee  -> (3, 3) : transition vers le play cooperatif : ATTEINTE 


**Lecture de la sortie commitee** : a matrice **fixee** dans la chambre Dilemme, la transition vers
le play cooperatif (3,3) est **inatteignable** au silence (l'equilibre de meilleure reponse reste
la defection mutuelle), **inatteignable** par la promesse seule (Colonne exploite la main liee :
(1,4)), **inatteignable** par la menace choisie (l'induction du sous-jeu de punition la retourne :
laisser domine punir, et Colonne anticipe) — et **atteinte** par la menace programmee observable
(3,3), la punition ne s'executant jamais : c'est une **dissuasion**. Le mur n'a pas ete franchi
dans l'atlas des 576 chambres — la matrice n'a pas bouge d'un rang — mais le **play** a traverse
ce que la forme normale tenait pour une frontiere. C'est l'epaisseur du mur : non pas une distance
dans le permutaedre, mais un ensemble d'arbres d'engagement capables ou non de conduire a l'issue.

In [13]:
// === Section 6.3 : la Chasse au Cerf -- la meme promesse, une autre epaisseur ===

var CC = CERF_CELLULES;

static Noeud ArbreCerfPromesse(Dictionary<(int, int), (int, int)> CC) {
    // Ligne se lie a traquer le cerf (C), engagement observable ; Colonne repond
    return Decision(1, new Dictionary<string, Noeud> {
        ["C"] = Feuille(CC[(0, 0)].Item1, CC[(0, 0)].Item2, "(C,C)"),
        ["D"] = Feuille(CC[(0, 1)].Item1, CC[(0, 1)].Item2, "(C,D)"),
    }, "cerf:J2-observe-C-force");
}

static Noeud ArbreCerfSilence(Dictionary<(int, int), (int, int)> CC) {
    (int, int) suiteSilence(int j2) {
        int meilleureJ1 = CC[(0, j2)].Item1 > CC[(1, j2)].Item1 ? 0 : 1;
        return CC[(meilleureJ1, j2)];
    }
    return Decision(1, new Dictionary<string, Noeud> {
        ["C"] = Feuille(suiteSilence(0).Item1, suiteSilence(0).Item2, "(BR,C)"),
        ["D"] = Feuille(suiteSilence(1).Item1, suiteSilence(1).Item2, "(BR,D)"),
    }, "cerf-silence:J2-jeu-de-base");
}

var sepInfoC = ", ";
var sepTraceC = " -> ";
var arbreSil = ArbreCerfSilence(CC);
var arbreProm = ArbreCerfPromesse(CC);
foreach (var (mode, arbre) in new[] { ("silence", arbreSil), ("promesse observable", arbreProm) }) {
    var u = ValeurSpe(arbre);
    Console.WriteLine($"--- Cerf / engagement = {mode}");
    Console.WriteLine($"    ensembles d'information : [{string.Join(sepInfoC, PartitionInformation(arbre))}]");
    foreach (var (tr, nom, uu) in CheminSpe(arbre))
        Console.WriteLine($"    chemin SPE {string.Join(sepTraceC, tr).PadRight(38)} issue {nom.PadRight(10)} utilites ({uu.Item1}, {uu.Item2})");
    Console.WriteLine($"    valeur SPE : ({u.Item1}, {u.Item2})");
}
Console.WriteLine();
Console.WriteLine($"Play cooperatif (C,C) du Cerf = ({CC[(0, 0)].Item1}, {CC[(0, 0)].Item2})");
Console.WriteLine("Au silence, le Cerf coordonne aussi sur (C,C) par meilleures reponses croisees :");
Console.WriteLine("la promesse y est AUTO-EXECUTOIRE (C est deja meilleure reponse a C), le mur du Cerf");
Console.WriteLine("etait poreux ; celle du Dilemme exigeait l'automatisation de la punition.");
Console.WriteLine("Memoriser : la MEME promesse, deux chambres, deux epaisseurs de mur.");


--- Cerf / engagement = silence


    ensembles d'information : [cerf-silence:J2-jeu-de-base]


    chemin SPE J2:C                                   issue (BR,C)     utilites (4, 4)


    valeur SPE : (4, 4)


--- Cerf / engagement = promesse observable


    ensembles d'information : [cerf:J2-observe-C-force]


... (3 outputs left out)

Au silence, le Cerf coordonne aussi sur (C,C) par meilleures reponses croisees :


la promesse y est AUTO-EXECUTOIRE (C est deja meilleure reponse a C), le mur du Cerf


etait poreux ; celle du Dilemme exigeait l'automatisation de la punition.


Memoriser : la MEME promesse, deux chambres, deux epaisseurs de mur.


**Lecture de la sortie commitee** : la promesse liee-au-cerf est **auto-executoire** — dans la
chambre Cerf, cooperer est deja la meilleure reponse a cooperer, l'engagement ne retire qu'un risk
de coordination, le play cooperatif etait atteignable sans lui. Dans la chambre Dilemme, la meme
promesse livree a Colonne ne vaut que le rang du dindon de la farce (1,4). **La forme extensive ne
deplace pas la chambre ; elle mesure la resistance du mur a chaque engin de traversree** — et cette
resistance est une propriete ordinale de la chambre (T>R>P>S vs R>T>P>S), donc un invariant de
l'atlas Robinson-Goforth lui-meme : deux chambres adjacentes peuvent porter des epaisseurs tres
differentes pour le meme engagement.

## 7. Ce que la loi autorise à dire — et ce qu'elle n'autorise pas

**Ce qui est acquis** : le passage du vérificateur au constructeur est fait sur un **second substrat**.
Étant donnés deux jeux, la machine **produit** la suite d'opérations élémentaires qui transforme
l'un en l'autre, et un vérificateur séparé **vérifie indépendamment** validité et minimalité par
énumération exhaustive de l'univers. Les trois verdicts du vérificateur sont distincts et opposables ; le
comptage des géodésiques rejoint la combinatoire de Coxeter (16 mots réduits de w₀ par table,
composés en 236 544 par entrelacement) ; et le résultat négatif du §5 est **prouvé par énumération
complète** : les murs ne raccourcissent aucune distance inter-chambres — le diamètre reste 12.
La loi n'était pas une spécificité de Life, et le substrat des jeux a livré une géométrie que le
seul permutaèdre ne montrait pas.

**Ce qui n'est PAS acquis** — les frontières honnêtes :

- **Ordinal 2×2 uniquement**. L'espace des jeux à payoffs cardinaux, ou même 3×3 ordinaux,
  n'est ni énuméré ni borné ici ; la vérification par exhaustivité meurt avec la finitude.
- **Le chemin est une géométrie, pas une dynamique**. Rien ne dit qu'un processus d'adaptation
  suiverait une géodésique ; les swaps ne sont pas des coups que des joueurs jouent.
- **La minimalité est relative au jeu de moves**. Avec swaps sur les deux tables, la distance
  d'un jeu est la somme des distances par table (12 pour le jeu, 6 par table) : le verdict
  énonce « minimal pour CES générateurs », jamais « le plus court possible dans l'absolu ».
- **Pas de jambe Lean dans ce grain**. La vérification est l'énumération exhaustive Python ; la
  formalisation `by decide` de la validité d'un pas est la variante naturelle suivante — la
  forme d'itération (`-b`, `-c`) que suit cette série.

**Sources** — Robinson & Goforth (1978), *The Topology of the 2×2 Games* (cité, RAPPORTÉ) ;
décomposition de Bruns-Kimmich, chambres / murs / `make_tie` : GT-3b ; le précédent
constructeur de ce type : le translateur Life (GOL-06). Chiffres structurels re-dérivés ici :
75, 576, 5625, diamètre 12 dans les deux univers, résultat du non-raccourci, géodésiques
16 / 2 / 236 544 (VÉRIFIÉS par ce notebook).

## Exercices

Les trois exercices utilisent le constructeur, le vérificateur et le compteur de géodésiques
définis plus haut. Le notebook s'exécute de bout en bout même sans les compléter.

### Exercice 1 — Le chemin retour

Construire le chemin **Poule → Dilemme** et le soumettre au vérificateur séparé. Vérifier que le
constructeur produit bien le même nombre de pas qu'à l'aller — et expliquer pourquoi la distance est
symétrique.

In [14]:
// Exercice 1 : chemin retour Poule -> Dilemme, construit puis verifie independamment.
// Etape 1 : chemin_retour = ConstruireChemin(POULE, PD, g => adjChambres[g])
// Etape 2 : verdict = VerifierChemin(POULE, PD, chemin_retour, adjChambres, g => adjChambres[g])
// Indice : la symetrie vient de ce que chaque swap est sa propre reciproque.
List<Tuple<Tuple<string, string>, Tuple<string, string>>> cheminRetour = null; // TODO etudiant
Console.WriteLine("Exercice 1 : a completer (chemin retour Poule -> Dilemme)");


Exercice 1 : a completer (chemin retour Poule -> Dilemme)


### Exercice 2 — Les géodésiques d'une autre paire

Compter les géodésiques **Dilemme → Chasse au Cerf** dans l'univers complet (murs autorisés),
et comparer au compte dans les chambres seules. Le mur crée-t-il de nouveaux mondes minimaux ?

In [15]:
// Exercice 2 : geodesiques PD -> Cerf, univers complet vs chambres.
// Etape 1 : nb_geodesiques_complet = CompterGeodesiques(PD, CERF, VoisinsComplets)
// Etape 2 : nb_geodesiques_chambres = CompterGeodesiques(PD, CERF, g => adjChambres[g])
// Indice : si la distance change entre les deux univers, le compte change aussi.
int nbGeodesiquesComplet = 0;    // TODO etudiant
int nbGeodesiquesChambres = 0;   // TODO etudiant
Console.WriteLine("Exercice 2 : a completer (geodesiques PD -> Cerf, deux univers)");


Exercice 2 : a completer (geodesiques PD -> Cerf, deux univers)


### Exercice 3 — Un couple antipodal de l'univers complet

Trouver un jeu **H** à distance maximale du Dilemme dans l'univers complet (5625), construire le
chemin, et le soumettre au vérificateur séparé. Indice : la distribution des distances du §5.1 dit
quelle couche chercher.

In [16]:
// Exercice 3 : antipode du Dilemme dans l'univers complet (5625).
// Etape 1 : recalculer BfsComplet(PD, VoisinsComplets), chercher les sommets de distance max.
// Etape 2 : ConstruireChemin + VerifierChemin vers l'un d'eux.
// Indice : l'antipode n'est pas unique -- la couche maximale compte plusieurs sommets.
Tuple<string, string> antipodePd = null; // TODO etudiant : le jeu (row, col) a distance maximale de PD
Console.WriteLine("Exercice 3 : a completer (antipode du Dilemme dans l'univers complet)");


Exercice 3 : a completer (antipode du Dilemme dans l'univers complet)


### Exercice 4 — La Poule en forme extensive

La chambre Poule (T>R>S>P) a deux equilibres purs en strategies : (C,D) et (D,C). Un eleve de
Robinson-Goforth propose : « Ligne s'engage OBSERVABLEMENT a faire le mort (se lier a D) ; Colonne,
voyant la roue abandonnee, fait pareil par prudence... non, s'ecarte ! »

Construisez l'arbre `arbre_poule_engagement_D` (Ligne liee a D, observable, Colonne repond),
executeriez l'induction, et comparez a (C,C). Le mur de la Poule est-il poreux a cet engin-là ?
Quel engagement rendrait (C,C) atteignable — et pourquoi la reponse est-elle plus difficile
que pour le Cerf ?

In [17]:
// Exercice 4 : Poule en forme extensive -- Ligne liee a D, observable.
// Indice : reutilisez ExtraireCellule(POULE, i, j), Decision(), ValeurSpe(), CheminSpe().
// Etape 1 : construire POULE_CELLULES.
// Etape 2 : construire l'arbre (branche D de Ligne retiree, Colonne observe).
// Etape 3 : verdict -- valeur SPE vs (C,C), chemin d'equilibre.
Dictionary<(int, int), (int, int)> POULE_CELLULES = null;  // TODO etudiant
Noeud arbrePouleEngagementD = null;                          // TODO etudiant
Console.WriteLine("Exercice a completer : verdict de l'engagement D observable sur la Poule.");


Exercice a completer : verdict de l'engagement D observable sur la Poule.


## À retenir

- **La loi II est franchie sur un second substrat** : le constructeur *produit* la suite d'échanges élémentaires qui transforme un jeu en un autre, et un **vérificateur séparé** — qui ne réutilise aucun état du constructeur — *vérifie indépendamment* validité ET minimalité par énumération exhaustive de l'univers fini.
- **Le comptage des géodésiques rejoint la combinatoire de Coxeter** : 16 géodésiques pour le renversement d'une table, exactement le nombre de mots réduits de l'élément le plus long `w₀` de `S₄` ; le DAG des couches BFS ne calcule rien d'autre que ces expressions réduites.
- **Résultat négatif prouvé par énumération complète** : les murs (jeux à égalités) ne raccourcissent AUCUNE distance inter-chambres — le diamètre reste 12, identique au permutaèdre des chambres. C'est un fait prouvé, pas l'absence de contre-exemple trouvé.

## Vérifiez votre compréhension

1. Pourquoi le vérificateur doit-il être *séparé* du constructeur (ne réutiliser aucun de ses états) ? Que permet chacun des trois verdicts distincts (VALIDE+MINIMAL / VALIDE+NON MINIMAL / INVALIDE) d'exclure ?
2. Le comptage donne 16 géodésiques pour l'antipode d'une table. Quelle prédiction externe de la théorie des groupes de Coxeter rejoint-il, et qu'aurait signifié un écart ?
3. En quoi « les murs ne raccourcissent rien » est-il un résultat *prouvé* et non une absence de contre-exemple ? Quel rôle joue la structure produit `S₄ × S₄` dans la portée de l'énumération ?

<details>
<summary>Éléments de réponse</summary>

1. Un constructeur ne peut pas se contrôler lui-même : le BFS qui trouve le chemin s'appuie sur ses propres structures (parents, table d'adjacence), et s'il vérifiait son propre résultat, une erreur de ces structures serait reproduite à l'identique — le contrôle ne distinguerait pas un chemin valide d'un chemin fabriqué par la même erreur. Le vérificateur du §4 ne réutilise rien : il re-dérive l'élémentarité de chaque pas contre le graphe, contrôle les extrémités, puis refait un BFS complet pour la distance. Les trois verdicts séparent trois défauts. VALIDE+MINIMAL exclut à la fois le pas non élémentaire et le détour : la sortie imprime `VALIDE + MINIMAL : 2 pas = distance exhaustive d(G,H)` pour le chemin du constructeur, et le même verdict sur l'univers complet (`5 pas`). VALIDE mais NON MINIMAL isole le cas « le chemin marche mais un plus court existe » : `4 pas, distance reelle 2`. INVALIDE rejette un pas qui n'est pas un move élémentaire ou des extrémités fausses : `pas non elementaire vers ((4, 1, 3, 2), (4, 3, 1, 2))` pour le saut PD → Cerf, deux chambres non adjacentes. Un vérificateur qui ne saurait dire que « valide » serait un tampon ; c'est le troisième verdict qui lui donne les dents.
2. La prédiction externe est le nombre de mots réduits de l'élément le plus long `w₀` de `S₄` : **16**. La sortie le mesure et le nomme — `Geodesiques (1,2,3,4) -> (4,3,2,1) sur une table : 16 (mots reduits de w0 dans S4 : 16 attendus)`. Le DAG des couches ne calcule rien d'autre que ces expressions réduites : une géodésique est une suite de swaps adjacents sans pas inutile, donc un mot réduit. L'intérêt de la prédiction est qu'elle est **externe** : elle vient de la combinatoire de Coxeter, pas d'une ligne du carnet, et la mesure peut la contredire. Un écart aurait donc signalé que l'instrument ne mesure pas ce qu'il prétend — générateurs qui ne sont pas les transpositions adjacentes standard de `S₄`, ou comptage par couches faux — et c'est la mesure qui aurait été réfutée. La mesure tombe juste, et la composition au jeu complet le confirme à sa manière : `236544` géodésiques pour l'antipode des deux tables, soit 16 × 16 × 924, les deux familles de seize mots réduits entrelacées par les `C(12,6) = 924` ordres de pas.
3. Parce que l'univers est **fini et épuisé**, pas échantillonné. Le §5.2 compare une à une les `24 × 24 = 576` paires d'ordres stricts dans le permutaèdre et dans l'espace à rangs complet, et la sortie donne `Paires ou l'espace complet (murs autorises) est PLUS COURT que le permutaedre : 0`. La structure produit fait le reste : chaque générateur ne touche qu'un seul côté, donc `d_jeu(G,H) = d_cote(row_G,row_H) + d_cote(col_G,col_H)` dans **chaque** univers ; le verdict par côté couvre alors `576 × 576 = 331776` paires de chambres sans exception, comme l'imprime la même sortie. Le diamètre le confirme : l'univers complet est connexe sur `5625 sommets` et garde `Diametre de l'univers complet : 12`, identique au permutaèdre des chambres. Un 0 trouvé sur un échantillon n'aurait rien prouvé ; ici l'énumération épuise l'espace et la décomposition par produit transporte le résultat du côté au jeu entier — c'est une preuve par exhaustivité, pas l'absence de contre-exemple trouvé.

</details>

## Pour aller plus loin

- **Le substrat Life (GOL-01)** — là où la loi II a été franchie pour la première fois (la machine *redécouvre* le glider au lieu de le recopier).
- **Le troisième substrat, 3×2 (Metajeux-06)** — où les murs *raccourcissent* bien : le contraste avec le résultat négatif d'ici.
- **L'épaisseur des murs en forme extensive** (§6) : engagements observables (promesse, menace programmée) à la manière de Schelling.